In [1]:
import sqlite3
from backend.engine.yahoo_transit import *
from backend.engine.database import *

In [2]:
dbpath = "../backend/engine/cache.db"
conn = get_conn(dbpath)

In [3]:
from engine.bus import *
from engine.train import *
from typing import Optional
# データの読み込み
dataset: dict[TransitType, list[Station]] = {
    TransitType.BUS: load_stop_data("../dataset/busstops/kanagawa/P11-22_14.geojson"),
    TransitType.TRAIN: load_station_data("../dataset/stations/N02-20_Station.geojson"),
}

# 小田急本厚木, JR新宿
honatugi_station: Optional[Station] = None
shinjuku_station: Optional[Station] = None

for stop in dataset[TransitType.TRAIN]:
    if stop.name == "本厚木":
        honatugi_station = stop
    if stop.name == "新宿":
        shinjuku_station = stop
if honatugi_station is None:
    print("本厚木駅をみつけられませんでした")
if shinjuku_station is None:
    print("新宿駅をみつけられませんでした")

In [4]:
routes = get_route_yahoo_transit(
    transit_type=TransitType.TRAIN,
    from_=shinjuku_station,
    to=honatugi_station
)
print(routes)

[{'time_required': 43, 'transfer': 0, 'fare': 513, 'distance': 45.4}, {'time_required': 48, 'transfer': 2, 'fare': 779, 'distance': 46.9}, {'time_required': 46, 'transfer': 2, 'fare': 837, 'distance': 46.3}]


In [8]:
for route in routes:
    req = InsertRouteReq(
        is_bus_route=False,
        from_=shinjuku_station,to_=honatugi_station,
        time_required=route["time_required"],
        transfer=route["transfer"],
        fare=route["fare"],
        distance=route["distance"],
    )
    insert_route(conn, req)

In [9]:
req = GetRoutesReq(
    is_bus_route=False,
    from_=shinjuku_station,to_=honatugi_station
)
routes = get_routes(conn, req)
print(routes)

[{'time_required': 43, 'transfer': 0, 'fare': 513, 'distance': 45}, {'time_required': 46, 'transfer': 2, 'fare': 837, 'distance': 46}, {'time_required': 48, 'transfer': 2, 'fare': 779, 'distance': 46}]
